# Automated GRPO Threshold Ladder Colab Runner

Automated Colab notebook for the single-session post-refactor GRPO threshold ladder.

This notebook:
- bootstraps the repo and checkpoints like `chess_model_run_git.ipynb`
- runs the three approved Stage 1 probes with checkpointed 20/30/40 epoch boundaries
- assesses exact run histories using WandB API `scan_history` on the exact run just launched
- writes structured JSON decision records after each ladder boundary for recovery
- skips or falls back conservatively when later stages are ambiguous or failed
- sizes Stage 3 from actual remaining session time instead of forcing a long run late in the session


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "search_refactor"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/data/grpo-chess"  #@param {type:"string"}
BASE_CHECKPOINT_PATH = "/content/drive/MyDrive/data/grpo-chess/base/9M.pt"  #@param {type:"string"}
RUN_NAME_PREFIX = "grpo-threshold-ladder"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
WANDB_ENTITY = ""  #@param {type:"string"}
SESSION_HOURS = 25.0  #@param {type:"number"}
ESTIMATED_MIN_PER_EPOCH = 2.80  #@param {type:"number"}
STAGE3_TARGET_EPOCHS = 300  #@param {type:"integer"}
SESSION_RESERVE_MINUTES = 30  #@param {type:"integer"}


In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("This notebook is intended to run in Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

repo = Path("/content/grpo_chess")
os.chdir("/content")
if repo.exists():
    shutil.rmtree(repo)

!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git checkout {REPO_REF}
!git submodule update --init --recursive

if str(repo) not in sys.path:
    sys.path.append(str(repo))

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
runtime_dir = drive_root / "colab_runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")


In [ ]:
import os
import signal
from pathlib import Path

deps_ready = Path("/tmp/grpo_reasoning_colab_deps_ready")
if not deps_ready.exists():
    %pip install -q --upgrade pip setuptools wheel
    !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
    %pip install -q -r /tmp/requirements-colab.txt
    !apt-get -qq update
    !apt-get -qq install -y stockfish
    !COLAB=1 bash scripts/setup_distill_deps.sh --checkpoint 9M --skip-checkpoint
    %pip install -q --force-reinstall --no-cache-dir pillow
    deps_ready.touch()
    print("Dependencies installed. Restarting runtime to load fresh binary modules...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Dependencies already installed for this runtime.")


In [ ]:
import subprocess
import urllib.request
import yaml
import zipfile
from pathlib import Path

base_checkpoint = Path(BASE_CHECKPOINT_PATH).expanduser()
base_checkpoint.parent.mkdir(parents=True, exist_ok=True)

searchless_ckpt_root = drive_root / "searchless_checkpoints"
searchless_ckpt_root.mkdir(parents=True, exist_ok=True)
repo_searchless_ckpt_root = repo / "searchless_chess" / "checkpoints"
repo_searchless_ckpt_root.mkdir(parents=True, exist_ok=True)
download_url = "https://storage.googleapis.com/searchless_chess/checkpoints/9M.zip"
zip_path = searchless_ckpt_root / "9M.zip"
nine_m_dir = searchless_ckpt_root / "9M"
teacher_download_url = "https://storage.googleapis.com/searchless_chess/checkpoints/136M.zip"
teacher_zip_path = searchless_ckpt_root / "136M.zip"
teacher_ckpt_dir = searchless_ckpt_root / "136M"

if not base_checkpoint.exists():
    if not nine_m_dir.exists():
        print("Downloading", download_url)
        urllib.request.urlretrieve(download_url, zip_path)
        print("Extracting", zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(searchless_ckpt_root)
        if zip_path.exists():
            zip_path.unlink()
    else:
        print("Reusing existing 9M checkpoint directory:", nine_m_dir)

    expected_orbax = nine_m_dir / "6400000" / "params" / "checkpoint"
    if not expected_orbax.exists():
        raise FileNotFoundError(f"Missing expected Orbax checkpoint file: {expected_orbax}")

    print("Converting downloaded 9M checkpoint to DM-port format:", base_checkpoint)
    conversion = subprocess.run(
        [
            sys.executable,
            "-m",
            "src.dm_port.convert_jax",
            "--model",
            "9M",
            "--checkpoint-dir",
            str(searchless_ckpt_root),
            "--out",
            str(base_checkpoint),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if conversion.stdout:
        print("Converter stdout:
", conversion.stdout)
    if conversion.stderr:
        print("Converter stderr:
", conversion.stderr)
    if conversion.returncode != 0:
        raise RuntimeError(
            f"DM checkpoint conversion failed with exit code {conversion.returncode}. "
            f"See printed converter stdout/stderr above."
        )

if not teacher_ckpt_dir.exists():
    print("Downloading", teacher_download_url)
    urllib.request.urlretrieve(teacher_download_url, teacher_zip_path)
    print("Extracting", teacher_zip_path)
    with zipfile.ZipFile(teacher_zip_path, "r") as zf:
        zf.extractall(searchless_ckpt_root)
    if teacher_zip_path.exists():
        teacher_zip_path.unlink()
else:
    print("Reusing existing 136M checkpoint directory:", teacher_ckpt_dir)

expected_teacher_orbax = teacher_ckpt_dir / "6400000" / "params" / "checkpoint"
if not expected_teacher_orbax.exists():
    raise FileNotFoundError(f"Missing expected 136M Orbax checkpoint file: {expected_teacher_orbax}")

repo_teacher_ckpt = repo_searchless_ckpt_root / "136M"
if repo_teacher_ckpt.exists() or repo_teacher_ckpt.is_symlink():
    repo_teacher_ckpt.unlink()
repo_teacher_ckpt.symlink_to(teacher_ckpt_dir, target_is_directory=True)
print("136M teacher checkpoint ready at:", repo_teacher_ckpt)


In [ ]:
import copy
import dataclasses
import json
import math
import os
import statistics
import time
from pathlib import Path

import torch
import wandb
import yaml

if USE_WANDB:
    from google.colab import userdata

    if WANDB_API_KEY.strip():
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    else:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    !wandb login
else:
    os.environ["WANDB_DISABLED"] = "true"

from src.colab_experiment_ladder import (
    HistoryWindow,
    RunAssessment,
    assess_recent_history,
    build_failed_assessment,
    choose_confirmation_source,
    compute_confirmation_epochs,
    select_best_stage1_run,
    select_stage2_bracket,
    should_skip_stage2,
)
from src.configs.config_loader import load_experiment_config
from src.train_self_play import train as grpo_train
import src.trainer as trainer_module

STAGE_1_CONFIGS = {
    "A1": "grpo_colab_probe_a1_lr3e6_kl5e4.yaml",
    "A2": "grpo_colab_probe_a2_lr5e6_kl3e4.yaml",
    "A3": "grpo_colab_probe_a3_lr8e6_kl1e4.yaml",
}
STAGE_1_BOUNDARIES = [20, 30, 40]
SESSION_START_TS = time.time()
LAUNCHED_RUNS = {}
SESSION_ROOT = drive_root / "runs" / RUN_NAME_PREFIX
SESSION_ROOT.mkdir(parents=True, exist_ok=True)
DECISION_RECORD_DIR = runtime_dir / f"{RUN_NAME_PREFIX}-decision-records"
DECISION_RECORD_DIR.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_PATH = runtime_dir / f"{RUN_NAME_PREFIX}-summary.json"
decision_records = []
failed_runs = []


class CapturingWandbLogger(trainer_module.WandbLogger):
    def __init__(self, *args, **kwargs):
        self._forced_run_name = kwargs.get("name", "")
        super().__init__(*args, **kwargs)
        experiment = self.experiment
        LAUNCHED_RUNS[self._forced_run_name] = {
            "run_id": getattr(experiment, "id", ""),
            "run_name": getattr(experiment, "name", self._forced_run_name),
        }




def _read_yaml(path: Path):
    return yaml.safe_load(path.read_text())


def _runtime_config_path(run_label: str) -> Path:
    return runtime_dir / f"{run_label}.yaml"


def _stage_root(stage_name: str) -> Path:
    root = SESSION_ROOT / stage_name.lower()
    root.mkdir(parents=True, exist_ok=True)
    return root


def _stage_checkpoint(stage_name: str) -> Path | None:
    ckpt = _stage_root(stage_name) / "last.ckpt"
    return ckpt if ckpt.exists() else None


def _record_decision(stage: str, event: str, payload: dict) -> None:
    record = {
        "timestamp": time.time(),
        "stage": stage,
        "event": event,
        **payload,
    }
    decision_records.append(record)
    filename = f"{len(decision_records):03d}-{stage.lower()}-{event}.json"
    (DECISION_RECORD_DIR / filename).write_text(json.dumps(record, indent=2, sort_keys=True))
    print("Decision record:", DECISION_RECORD_DIR / filename)


def _build_runtime_config(config_filename: str, stage_name: str, run_label: str, num_epochs: int) -> Path:
    config_path = Path("src/configs") / config_filename
    config_data = copy.deepcopy(_read_yaml(config_path))
    config_data["model"]["base_checkpoint"] = str(base_checkpoint)
    config_data["rival"]["frozen_dm_9m"]["checkpoint_path"] = str(base_checkpoint)
    config_data["training"]["checkpoint_dir"] = str(_stage_root(stage_name))
    config_data["training"]["use_wandb"] = bool(USE_WANDB)
    config_data["training"]["num_epochs"] = int(num_epochs)
    runtime_config_path = _runtime_config_path(run_label)
    runtime_config_path.write_text(yaml.safe_dump(config_data, sort_keys=False))
    return runtime_config_path


def _wandb_entity(api: wandb.Api) -> str:
    if WANDB_ENTITY.strip():
        return WANDB_ENTITY.strip()
    entity = os.environ.get("WANDB_ENTITY") or getattr(api, "default_entity", None)
    if entity:
        return entity
    raise RuntimeError("Could not resolve WANDB entity. Set WANDB_ENTITY explicitly in the notebook parameters.")


def _resolve_exact_run(project: str, run_name: str, run_id: str = "", *, retries: int = 6, delay_seconds: int = 10):
    if not USE_WANDB:
        raise RuntimeError("Exact run resolution requires WandB to be enabled.")
    last_error = None
    for attempt in range(retries):
        try:
            api = wandb.Api()
            entity = _wandb_entity(api)
            if run_id:
                run = api.run(f"{entity}/{project}/{run_id}")
                display_name = getattr(run, "display_name", None) or getattr(run, "name", None)
                if display_name == run_name or getattr(run, "name", None) == run_name:
                    return run
                last_error = RuntimeError(
                    f"Resolved run id {run_id} but got name={display_name}, expected {run_name}"
                )
                raise last_error
            runs = api.runs(f"{entity}/{project}", per_page=100, order="-created_at")
            for run in runs:
                display_name = getattr(run, "display_name", None) or getattr(run, "name", None)
                if display_name == run_name or getattr(run, "name", None) == run_name:
                    return run
            last_error = RuntimeError(f"Exact WandB run not found for name={run_name}")
        except Exception as exc:
            last_error = exc
        if attempt + 1 < retries:
            print(f"Waiting for WandB run {run_name} to appear ({attempt + 1}/{retries})...")
            time.sleep(delay_seconds)
    raise RuntimeError(f"Failed to resolve exact WandB run {run_name}: {last_error}")


def _finite_values(rows: list[dict], key: str) -> list[float]:
    values = []
    for row in rows:
        value = row.get(key)
        if value is None:
            continue
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)
    return values


def _summarize_history_window(rows: list[dict], eval_score: float) -> HistoryWindow:
    ratios = _finite_values(rows, "train/ratio_step")
    clips = _finite_values(rows, "train/clip_fraction_step")
    ppo = [abs(v) for v in _finite_values(rows, "train/ppo_loss_step")]
    kl = [abs(v) for v in _finite_values(rows, "train/kl_divergence_step")]
    losses = _finite_values(rows, "train_total_loss_step")
    if not ratios or not clips or not ppo or not kl or not losses:
        raise RuntimeError("Recent WandB history window is missing required metrics for assessment")

    def _mean(values: list[float]) -> float:
        return sum(values) / len(values)

    def _std(values: list[float]) -> float:
        if len(values) < 2:
            return 0.0
        return statistics.pstdev(values)

    return HistoryWindow(
        ratio_mean=_mean(ratios),
        ratio_std=_std(ratios),
        clip_fraction_mean=_mean(clips),
        clip_fraction_peak=max(clips),
        ppo_abs_mean=_mean(ppo),
        kl_abs_mean=_mean(kl),
        loss_mean=_mean(losses),
        loss_std=_std(losses),
        eval_score=eval_score,
        point_count=min(len(ratios), len(clips), len(ppo), len(kl), len(losses)),
    )


def _assess_exact_run(run_name: str, config_name: str, epochs_completed: int, project: str, run_id: str = "") -> RunAssessment:
    api_run = _resolve_exact_run(project, run_name, run_id=run_id)
    keys = [
        "train/ratio_step",
        "train/clip_fraction_step",
        "train/ppo_loss_step",
        "train/kl_divergence_step",
        "train_total_loss_step",
        "eval_stockfish/score",
    ]
    rows = list(api_run.scan_history(keys=keys, page_size=200))
    if not rows:
        raise RuntimeError(f"No WandB history returned for run {run_name} ({api_run.id})")
    recent_rows = rows[-12:]
    eval_history = _finite_values(rows, "eval_stockfish/score")
    summary_eval = float(api_run.summary.get("eval_stockfish/score", 0.0) or 0.0)
    window = _summarize_history_window(recent_rows, eval_history[-1] if eval_history else summary_eval)
    assessment = assess_recent_history(
        run_name=run_name,
        config_name=config_name,
        run_id=api_run.id,
        window=window,
        epochs_completed=epochs_completed,
    )
    return assessment


def _run_segment(stage_name: str, config_filename: str, *, target_epochs: int, resume_from_checkpoint: str | None = None) -> RunAssessment:
    run_label = f"{RUN_NAME_PREFIX}-{stage_name.lower()}-e{target_epochs}"
    runtime_config_path = _build_runtime_config(config_filename, stage_name, run_label, num_epochs=target_epochs)
    cfg = load_experiment_config(runtime_config_path)
    original_generate_run_name = trainer_module.generate_run_name
    original_wandb_logger = trainer_module.WandbLogger
    trainer_module.generate_run_name = lambda project="chess-grpo": run_label
    trainer_module.WandbLogger = CapturingWandbLogger
    start_ts = time.time()
    try:
        wandb.finish()
        grpo_train(
            config_path=str(runtime_config_path),
            resume_from_checkpoint=str(resume_from_checkpoint) if resume_from_checkpoint else None,
        )
        elapsed_minutes = (time.time() - start_ts) / 60.0
        launched_run = LAUNCHED_RUNS.get(run_label, {})
        assessment = _assess_exact_run(
            run_label,
            config_filename,
            target_epochs,
            cfg.training.wandb_project,
            run_id=launched_run.get("run_id", ""),
        )
        _record_decision(
            stage_name,
            f"segment_e{target_epochs}",
            {
                "run_name": run_label,
                "captured_run": launched_run,
                "run_id": assessment.run_id,
                "config_name": config_filename,
                "epochs_completed": target_epochs,
                "elapsed_minutes": elapsed_minutes,
                "assessment": dataclasses.asdict(assessment),
            },
        )
        return assessment
    except Exception as exc:
        elapsed_minutes = (time.time() - start_ts) / 60.0
        failed = build_failed_assessment(
            run_name=run_label,
            config_name=config_filename,
            epochs_completed=target_epochs,
            reason=str(exc),
        )
        failed_runs.append(dataclasses.asdict(failed))
        _record_decision(
            stage_name,
            f"segment_e{target_epochs}_failed",
            {
                "run_name": run_label,
                "config_name": config_filename,
                "epochs_completed": target_epochs,
                "elapsed_minutes": elapsed_minutes,
                "error": str(exc),
            },
        )
        return failed
    finally:
        trainer_module.generate_run_name = original_generate_run_name
        trainer_module.WandbLogger = original_wandb_logger
        wandb.finish()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_experiment(stage_name: str, config_filename: str, *, target_epochs: int, resume_from_checkpoint: str | None = None) -> RunAssessment:
    return _run_segment(
        stage_name,
        config_filename,
        target_epochs=target_epochs,
        resume_from_checkpoint=resume_from_checkpoint,
    )


def _run_stage1_probe(stage_name: str, config_filename: str) -> RunAssessment:
    last_assessment = None
    resume_checkpoint = None
    for boundary in STAGE_1_BOUNDARIES:
        assessment = run_experiment(
            stage_name,
            config_filename,
            target_epochs=boundary,
            resume_from_checkpoint=resume_checkpoint,
        )
        last_assessment = assessment
        if assessment.failed:
            return assessment
        if boundary < STAGE_1_BOUNDARIES[-1] and assessment.stop_reason in {"dead_run", "instability"}:
            assessment = dataclasses.replace(
                assessment,
                decision_reason=f"Stopped at epoch {boundary} because {assessment.stop_reason}",
            )
            _record_decision(
                stage_name,
                "probe_stopped_early",
                {
                    "winner": stage_name,
                    "config_name": config_filename,
                    "epochs_completed": boundary,
                    "stop_reason": assessment.stop_reason,
                    "assessment": dataclasses.asdict(assessment),
                },
            )
            return assessment
        resume_checkpoint = _stage_checkpoint(stage_name)
        if boundary < STAGE_1_BOUNDARIES[-1] and resume_checkpoint is None:
            failed = build_failed_assessment(
                run_name=stage_name,
                config_name=config_filename,
                epochs_completed=boundary,
                reason="missing resume checkpoint after completed segment",
            )
            failed_runs.append(dataclasses.asdict(failed))
            _record_decision(
                stage_name,
                "missing_resume_checkpoint",
                {
                    "config_name": config_filename,
                    "epochs_completed": boundary,
                    "reason": failed.decision_reason,
                    "assessment": dataclasses.asdict(failed),
                },
            )
            return failed
    return last_assessment


def _remaining_hours() -> float:
    elapsed_hours = (time.time() - SESSION_START_TS) / 3600.0
    return max(0.0, SESSION_HOURS - elapsed_hours)


def _write_summary(payload: dict) -> None:
    RUN_SUMMARY_PATH.write_text(json.dumps(payload, indent=2, sort_keys=True))
    print("Wrote ladder summary to", RUN_SUMMARY_PATH)


In [ ]:
from dataclasses import asdict, replace

stage1_results = {}
for stage_name, config_filename in STAGE_1_CONFIGS.items():
    stage1_results[stage_name] = _run_stage1_probe(stage_name, config_filename)

valid_stage1 = {name: result for name, result in stage1_results.items() if not result.failed}
if not valid_stage1:
    raise RuntimeError("All Stage 1 probe runs failed; ladder cannot continue safely.")

best_stage1 = select_best_stage1_run(stage1_results)
rejected_stage1 = {
    name: {
        "reason": result.stop_reason or result.decision_reason or "not selected",
        "assessment": asdict(result),
    }
    for name, result in stage1_results.items()
    if name != best_stage1
}
_record_decision(
    "STAGE1",
    "winner_selected",
    {
        "winner": best_stage1,
        "winner_assessment": asdict(stage1_results[best_stage1]),
        "rejected_candidates": rejected_stage1,
    },
)

stage2_skipped = should_skip_stage2(stage1_results, best_stage1=best_stage1)
stage2_result = None
stage2_config = None
if stage2_skipped:
    _record_decision(
        "STAGE2",
        "skipped",
        {
            "winner": best_stage1,
            "reason": "clear dominant Stage 1 winner with no useful surviving neighbor signal",
        },
    )
else:
    stage2_config = select_stage2_bracket(stage1_results, best_stage1=best_stage1)
    stage2_result = run_experiment("B1", stage2_config, target_epochs=100, resume_from_checkpoint=None)
    if stage2_result.failed:
        stage2_result = replace(stage2_result, inconclusive=True)
    else:
        best_stage1_assessment = stage1_results[best_stage1]
        movement_gain = stage2_result.movement_score - best_stage1_assessment.movement_score
        eval_gain = stage2_result.eval_score - best_stage1_assessment.eval_score
        # Movement score combines ratio deviation, clip fraction, PPO magnitude, and KL magnitude
        # into an approximately 0-3 range. Gains below ~0.15 are usually too small to trust.
        stage2_inconclusive = (
            stage2_result.unstable
            or not stage2_result.stable
            or movement_gain < 0.15
            or (movement_gain < 0.08 and eval_gain < 0.01)
        )
        stage2_reason = (
            "inconclusive midpoint bracket"
            if stage2_inconclusive
            else "bracket run is clearly better than Stage 1 winner"
        )
        stage2_result = replace(stage2_result, inconclusive=stage2_inconclusive, decision_reason=stage2_reason)
    _record_decision(
        "STAGE2",
        "completed",
        {
            "selected_config": stage2_config,
            "result": asdict(stage2_result),
            "fallback_used": bool(stage2_result.inconclusive if stage2_result else False),
        },
    )

confirmation_source = choose_confirmation_source(stage1_results[best_stage1], stage2_result)
if confirmation_source == "B1" and stage2_result is not None:
    confirmation_config = stage2_config
    confirmation_resume = _stage_checkpoint("B1")
    base_epochs_completed = stage2_result.epochs_completed
else:
    confirmation_config = STAGE_1_CONFIGS[confirmation_source]
    confirmation_resume = _stage_checkpoint(confirmation_source)
    base_epochs_completed = stage1_results[confirmation_source].epochs_completed

remaining_epochs = compute_confirmation_epochs(
    remaining_hours=_remaining_hours(),
    estimated_min_per_epoch=ESTIMATED_MIN_PER_EPOCH,
    reserve_minutes=SESSION_RESERVE_MINUTES,
    target_epochs=STAGE3_TARGET_EPOCHS,
)

_record_decision(
    "STAGE3",
    "launch_decision",
    {
        "confirmation_source": confirmation_source,
        "confirmation_config": confirmation_config,
        "remaining_hours": _remaining_hours(),
        "remaining_epochs": remaining_epochs,
        "base_epochs_completed": base_epochs_completed,
        "fallback_used": confirmation_source != "B1",
    },
)

confirmation_result = None
if remaining_epochs > 0 and confirmation_resume is not None:
    confirmation_result = run_experiment(
        "C1",
        confirmation_config,
        target_epochs=base_epochs_completed + remaining_epochs,
        resume_from_checkpoint=str(confirmation_resume),
    )
    _record_decision(
        "STAGE3",
        "failed" if confirmation_result.failed else "completed",
        {
            "confirmation_source": confirmation_source,
            "confirmation_config": confirmation_config,
            "result": asdict(confirmation_result),
            "fallback_used": confirmation_source != "B1",
        },
    )
else:
    _record_decision(
        "STAGE3",
        "skipped",
        {
            "reason": "not enough safe wall-clock budget left or no resume checkpoint available",
            "remaining_epochs": remaining_epochs,
            "resume_checkpoint_exists": confirmation_resume is not None,
        },
    )

summary_payload = {
    "stage1_results": {name: asdict(result) for name, result in stage1_results.items()},
    "best_stage1": best_stage1,
    "stage2_skipped": stage2_skipped,
    "stage2_config": stage2_config,
    "stage2_result": asdict(stage2_result) if stage2_result is not None else None,
    "confirmation_source": confirmation_source,
    "confirmation_config": confirmation_config,
    "remaining_confirmation_epochs": remaining_epochs,
    "confirmation_result": asdict(confirmation_result) if confirmation_result is not None else None,
    "failed_runs": failed_runs,
    "decision_record_dir": str(DECISION_RECORD_DIR),
}
_write_summary(summary_payload)
_record_decision("LADDER", "completed", summary_payload)
summary_payload
